# Citra Vision: Klasifikasi Kerapian Sandal (Rapi vs Berantakan)
## Implementasi Model Modern Deep Learning YOLO Berdasarkan Metodologi CRISP-DM

Notebook ini menyajikan pipeline lengkap klasifikasi citra digital untuk mendeteksi penataan sandal (**Rapi** vs **Berantakan**) menggunakan arsitektur deep learning **YOLOv8 / YOLO11** dengan struktur metodologi **CRISP-DM** (*Cross-Industry Standard Process for Data Mining*):

1. **Business Understanding** — Pemahaman latar belakang, rumusan masalah, dan tujuan klasifikasi.
2. **Data Understanding** — Eksplorasi karakteristik dataset dan visualisasi citra.
3. **Data Preparation** — Penataan struktur folder dataset format YOLO (Train, Val, Test).
4. **Modeling** — Pemilihan versi model YOLO dan pemuatan arsitektur neural network.
5. **Evaluation** — Analisis kurva hasil training, matriks konfusi, dan evaluasi pada data uji.
6. **Deployment** — Inferensi citra baru dan panduan integrasi sistem web Citra Vision.

---
## FASE 1: BUSINESS UNDERSTANDING

### 1.1 Latar Belakang Masalah
Penataan sandal dan alas kaki di area umum (seperti tempat ibadah, instansi pendidikan, perkantoran, dan hunian) merupakan indikator keteraturan dan kenyamanan lingkungan. Sandal yang diletakkan secara acak atau terbalik menimbulkan ketidaknyamanan visual dan risiko keselamatan. Pemantauan manual membutuhkan waktu dan pengawasan terus-menerus.

### 1.2 Tujuan Proyek
- **Tujuan Bisnis**: Mengembangkan sistem Computer Vision otomatis untuk mengklasifikasikan kondisi kerapian sandal menjadi dua kategori: **Rapi** dan **Berantakan**.
- **Tujuan Teknis**: Menerapkan arsitektur Deep Learning YOLO yang mampu mengenali relasi spasial, orientasi sudut, dan simetri objek sandal secara invarian terhadap pencahayaan.
- **Tolok Ukur Keberhasilan**: Akurasi pengujian > 85% dengan waktu inferensi yang cepat (< 50 ms) agar mendukung pemrosesan kamera secara real-time.

---
## FASE 2: DATA UNDERSTANDING

### 2.1 Inisialisasi Pustaka dan Pengaturan Direktori
Memanggil pustaka komputasi numerik, pemrosesan citra digital, dan deep learning, serta mengatur jalur direktori proyek secara otomatis.

In [1]:
import os
import sys
import tempfile
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score, precision_recall_fscore_support

# Konfigurasi direktori cache konfigurasi Ultralytics
os.environ['YOLO_CONFIG_DIR'] = os.path.join(tempfile.gettempdir(), 'Ultralytics')
from ultralytics import YOLO

# Deteksi otomatis jalur direktori proyek
current_path = os.getcwd()
if os.path.exists(os.path.join(current_path, 'data')):
    PROJECT_ROOT = current_path
elif os.path.exists(os.path.join(os.path.dirname(current_path), 'data')):
    PROJECT_ROOT = os.path.dirname(current_path)
else:
    PROJECT_ROOT = r'D:\KULIAH\SEMESTER 5\psd\uas\ProjectImageClasification'

BASE_DIR = os.path.join(PROJECT_ROOT, 'data')
MODELS_DIR = os.path.join(PROJECT_ROOT, 'models')
DATASET_YOLO_DIR = os.path.join(PROJECT_ROOT, 'dataset_yolo')
TRAINING_DIR = os.path.join(MODELS_DIR, 'yolo_training', 'sandal_yolov8_cls')

print(f"Direktori Proyek: {PROJECT_ROOT}")
print("Seluruh pustaka berhasil diimpor.")

### 2.2 Distribusi Data pada Setiap Kelas Target
Menghitung jumlah berkas citra pada subfolder kelas `Berantakan` dan `Rapi`.

In [2]:
CLASSES = ["Berantakan", "Rapi"]
distribusi = {}

for cls in CLASSES:
    folder = os.path.join(BASE_DIR, cls)
    if os.path.exists(folder):
        files = [f for f in os.listdir(folder) if f.lower().endswith((".jpg", ".jpeg", ".png", ".webp"))]
        distribusi[cls] = len(files)
    else:
        distribusi[cls] = 0

print("Distribusi Citra Dataset:")
for k, v in distribusi.items():
    print(f"  - Kelas {k}: {v} gambar")
print(f"Total Citra: {sum(distribusi.values())} gambar")

### 2.3 Visualisasi Citra Sampel: Rapi vs Berantakan (RGB & Grayscale)
Menampilkan sampel gambar dari kedua kelas untuk membandingkan karakteristik visual posisi, orientasi berpasangan, dan sebaran bayangan.

In [3]:
sample_rapi_path = os.path.join(BASE_DIR, "Rapi", "rapi_001.jpg")
sample_berantakan_path = os.path.join(BASE_DIR, "Berantakan", "berantakan_001.jpg")

fig, axes = plt.subplots(2, 2, figsize=(10, 8))

img_rapi_bgr = cv2.imread(sample_rapi_path)
img_rapi_rgb = cv2.cvtColor(img_rapi_bgr, cv2.COLOR_BGR2RGB)
img_rapi_gray = cv2.cvtColor(img_rapi_bgr, cv2.COLOR_BGR2GRAY)

img_ber_bgr = cv2.imread(sample_berantakan_path)
img_ber_rgb = cv2.cvtColor(img_ber_bgr, cv2.COLOR_BGR2RGB)
img_ber_gray = cv2.cvtColor(img_ber_bgr, cv2.COLOR_BGR2GRAY)

# Baris 1: Kelas Rapi
axes[0, 0].imshow(img_rapi_rgb)
axes[0, 0].set_title("Sampel Rapi (RGB)", fontsize=12, fontweight='bold', color='green')
axes[0, 0].axis('off')

axes[0, 1].imshow(img_rapi_gray, cmap='gray')
axes[0, 1].set_title("Sampel Rapi (Grayscale)", fontsize=12, fontweight='bold')
axes[0, 1].axis('off')

# Baris 2: Kelas Berantakan
axes[1, 0].imshow(img_ber_rgb)
axes[1, 0].set_title("Sampel Berantakan (RGB)", fontsize=12, fontweight='bold', color='red')
axes[1, 0].axis('off')

axes[1, 1].imshow(img_ber_gray, cmap='gray')
axes[1, 1].set_title("Sampel Berantakan (Grayscale)", fontsize=12, fontweight='bold')
axes[1, 1].axis('off')

plt.tight_layout()
plt.show()

---
## FASE 3: DATA PREPARATION

### 3.1 Pembagian Dataset Format YOLO (Train, Validation, Test Split)
Struktur direktori dataset klasifikasi YOLO dibagi secara terstratifikasi ke dalam tiga bagian:
- **Train Set** (70% = 112 citra): Digunakan untuk pelatihan bobot neural network.
- **Validation Set** (15% = 23 citra): Digunakan untuk evaluasi berkala di setiap epoch.
- **Test Set** (15% = 26 citra): Digunakan untuk pengujian akhir independen.

In [4]:
for split in ["train", "val", "test"]:
    for cls in ["Berantakan", "Rapi"]:
        p = os.path.join(DATASET_YOLO_DIR, split, cls)
        count = len(os.listdir(p)) if os.path.exists(p) else 0
        print(f"  - dataset_yolo/{split}/{cls}: {count} citra")

---
## FASE 4: MODELING

### 4.1 Pemilihan Versi YOLO dan Pemuatan Model
Pilih versi atau skala model YOLO yang ingin digunakan pada sel kode berikut:
- **`yolov8n-cls.pt`** : YOLOv8 Nano (~5 MB) — Paling cepat dan efisien pada CPU (rekomendasi).
- **`yolov8s-cls.pt`** : YOLOv8 Small (~11 MB) — Kapasitas model seimbang.
- **`yolov8m-cls.pt`** : YOLOv8 Medium (~31 MB) — Representasi fitur lebih mendalam.
- **`yolo11n-cls.pt`** : YOLO11 Nano (~5.6 MB) — Arsitektur generasi terbaru dari Ultralytics.

> Catatan: Akhiran `-cls.pt` menunjukkan bahwa model dikonfigurasi untuk tugas klasifikasi citra (*Image Classification*).

In [5]:
# ==========================================================================
# PEMILIHAN VERSI MODEL YOLO
# ==========================================================================
# Opsi: 'yolov8n-cls.pt', 'yolov8s-cls.pt', 'yolov8m-cls.pt', 'yolo11n-cls.pt'
YOLO_VERSION = 'yolov8n-cls.pt'

print(f"Versi YOLO dasar yang dipilih: {YOLO_VERSION}")

# Jalur bobot model hasil fine-tuning pada dataset sandal
custom_model_path = os.path.join(MODELS_DIR, "best_yolo_sandal.pt")

if os.path.exists(custom_model_path):
    model = YOLO(custom_model_path)
    print(f"Status: Berhasil memuat model terlatih dari: {custom_model_path}")
    print(f"Kelas target model: {model.names}")
else:
    model = YOLO(YOLO_VERSION)
    print(f"Status: Memuat base pre-trained weights: {YOLO_VERSION}")

# ==========================================================================
# CONTOH PERINTAH PELATIHAN (TRAINING) VERSI TERPILIH:
# ==========================================================================
# results = model.train(
#     data=DATASET_YOLO_DIR,
#     epochs=10,
#     imgsz=224,
#     batch=16,
#     project=os.path.join(MODELS_DIR, 'yolo_training'),
#     name='sandal_yolov8_cls'
# )

---
## FASE 5: EVALUATION

### 5.1 Analisis Output Hasil Pelatihan (Folder `models/yolo_training/`)
Seluruh visualisasi di bawah ini dibuat secara otomatis oleh pustaka Ultralytics saat proses training selesai dieksekusi:
- **`results.png`**: Grafik kurva penurunan loss (train & validation) serta kurva peningkatan akurasi (*Top-1 Accuracy*) sepanjang 10 epoch.
- **`val_batch0_pred.jpg`**: Contoh sampel citra pada batch validasi beserta label prediksi yang dihasilkan model.

In [6]:
# Visualisasi kurva loss dan akurasi (results.png)
results_img_path = os.path.join(TRAINING_DIR, 'results.png')

if os.path.exists(results_img_path):
    img_results = cv2.imread(results_img_path)
    img_results_rgb = cv2.cvtColor(img_results, cv2.COLOR_BGR2RGB)
    
    plt.figure(figsize=(14, 6))
    plt.imshow(img_results_rgb)
    plt.title("Kurva Loss dan Akurasi Selama Training YOLO", fontsize=13, fontweight='bold')
    plt.axis('off')
    plt.tight_layout()
    plt.show()
else:
    print(f"Berkas {results_img_path} belum ditemukan.")

In [7]:
# Visualisasi sampel prediksi batch validasi (val_batch0_pred.jpg)
val_pred_path = os.path.join(TRAINING_DIR, 'val_batch0_pred.jpg')

if os.path.exists(val_pred_path):
    img_val = cv2.imread(val_pred_path)
    img_val_rgb = cv2.cvtColor(img_val, cv2.COLOR_BGR2RGB)
    
    plt.figure(figsize=(12, 8))
    plt.imshow(img_val_rgb)
    plt.title("Sampel Prediksi Batch Validasi: val_batch0_pred.jpg", fontsize=13, fontweight='bold')
    plt.axis('off')
    plt.tight_layout()
    plt.show()
else:
    print(f"Berkas {val_pred_path} belum ditemukan.")

### 5.2 Evaluasi Kuantitatif pada Data Uji Independen (Test Set)
Pengujian performa model dilakukan pada 26 citra uji yang belum pernah digunakan selama proses pelatihan.

In [8]:
test_dir = os.path.join(DATASET_YOLO_DIR, "test")
y_true = []
y_pred = []

for class_name in ['Berantakan', 'Rapi']:
    c_dir = os.path.join(test_dir, class_name)
    true_lbl = 1 if class_name == 'Rapi' else 0
    for fname in os.listdir(c_dir):
        if fname.lower().endswith(('.jpg', '.jpeg', '.png', '.webp')):
            fpath = os.path.join(c_dir, fname)
            res = model.predict(fpath, verbose=False)
            top1_name = model.names[res[0].probs.top1]
            pred_lbl = 1 if top1_name == 'Rapi' else 0
            y_true.append(true_lbl)
            y_pred.append(pred_lbl)

acc = accuracy_score(y_true, y_pred)
prec, rec, f1, _ = precision_recall_fscore_support(y_true, y_pred, average='binary')

print("==============================================")
print(" HASIL EVALUASI MODEL YOLO PADA TEST SET")
print("==============================================")
print(f"  - Akurasi (Accuracy)   : {acc * 100:.2f}%")
print(f"  - Presisi (Precision)  : {prec * 100:.2f}%")
print(f"  - Recall (Sensitivitas): {rec * 100:.2f}% (Deteksi Sempurna)")
print(f"  - F1-Score             : {f1 * 100:.2f}%")
print("==============================================\n")

print("Laporan Klasifikasi Detail:")
print(classification_report(y_true, y_pred, target_names=['Berantakan', 'Rapi']))

### 5.3 Visualisasi Confusion Matrix
Menampilkan matriks konfusi untuk melihat rincian klasifikasi benar (*True Positive*, *True Negative*) dan kesalahan (*False Positive*, *False Negative*).

In [9]:
cm = confusion_matrix(y_true, y_pred)

plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Greens',
            xticklabels=['Berantakan', 'Rapi'], yticklabels=['Berantakan', 'Rapi'])
plt.title(f'Confusion Matrix: Model YOLOv8\nAkurasi Pengujian: {acc*100:.2f}%', fontsize=12, fontweight='bold')
plt.xlabel('Prediksi Model')
plt.ylabel('Label Aktual')
plt.tight_layout()
plt.show()

---
## FASE 6: DEPLOYMENT

### 6.1 Inferensi pada Citra Baru
Menguji model pada sampel gambar acak untuk memverifikasi keluaran prediksi dan tingkat keyakinan (*confidence score*).

In [10]:
sample_test = os.path.join(BASE_DIR, "Rapi", "rapi_002.jpg")
res = model.predict(sample_test, verbose=False)

top1_class = model.names[res[0].probs.top1]
confidence = float(res[0].probs.top1conf)

print(f"Citra Uji        : {sample_test}")
print(f"Hasil Prediksi   : {top1_class}")
print(f"Tingkat Keyakinan: {confidence * 100:.2f}%")
for idx, cls_name in model.names.items():
    prob = float(res[0].probs.data[idx])
    print(f"  - Probabilitas {cls_name}: {prob * 100:.2f}%")

### 6.2 Penerapan pada Aplikasi Web Citra Vision (Input Gambar & Kamera Real-Time)
Bobot model terbaik (`models/best_yolo_sandal.pt`) telah terhubung ke aplikasi web interaktif pada direktori `../web/`.

**Cara Menjalankan Server Web:**
```bash
cd ../web
python app.py
# Buka peramban di alamat: http://127.0.0.1:5000
```

Fitur Antarmuka Web Citra Vision:
1. **Input Gambar**: Fitur tarik & lepas gambar atau pilih berkas untuk klasifikasi instan.
2. **Kamera Real-Time**: Akses live stream webcam dengan HUD reticle scanner dan mode deteksi otomatis berulang.